# 06 · Session-specific and covariance-aware pair feature laboratory

**A new, input-driven representation investigation: 41 candidate templates, zero model fits.** This notebook constructs candidates, checks prefix equivalence, measures supported coverage, and screens them **only on the first training prefix**. Do not interpret training correlations as predictive improvements. Run after notebook 05 completes successfully. The separate worker has a **90-second cap**. No packages, AWS resources, Git branches, or existing study source are modified.

Keep models and feature arrays private; do not delete old project directories, which hold the borrowed verified environment.

## Publication provenance

This is a publication copy, not a new execution. Original code cells: 10; cells with execution counts: 10; original error outputs: 0; inline Plotly outputs retained: 6. Console logs, table/HTML output, attachments and tracebacks were omitted from this copy. The original remains in the research workspace. An execution count alone is not proof of success. Refer to the linked result ledger for completed/failed/pending study status. Interactive figure data are disclosed with this public notebook.


In [1]:
from pathlib import Path
import os, importlib.util
import pandas as pd
from IPython.display import display, Markdown
ROOT = Path(os.environ.get('COMMODITY_MANUAL_PROJECT', '/home/sagemaker-user/projects/commodity-prediction-manual'))
spec = importlib.util.spec_from_file_location('manual_feature_round', ROOT / 'scripts/commodity_feature_round.py')
support = importlib.util.module_from_spec(spec)
spec.loader.exec_module(support)
previous = support.support(ROOT)
ready = previous.readiness(ROOT)
previous.validate_runtime(ROOT, ready)
print('Source:', ready['source_commit'])
print('Verified kernel. No new fitting has started in this cell.')


## Hypothesis and novelty check

The current features normalize each asset by close-to-close risk and then form pair differences. This experiment first forms the **actual signed pair intraday/overnight return**, then estimates its own past mean and volatility. That preserves each session’s covariance structure. The exact identity `Var(A − B) = Var(A) + Var(B) − 2 Cov(A,B)` motivates the pair representation.

Existing `domain/relationships.py` already has close-to-close pair correlations, spread risk and hedge innovations. **Those are not being relabeled as new.** The additions are component-specific surprises, relative session risk and explicit gap/reversal interactions. These names describe the supplied OHLC bars; they do not assert that international exchanges close simultaneously. All legs must have supported, valid bar definitions; otherwise an explicit support flag distinguishes structural absence from observed missingness.

## Candidate catalog: 41 templates

For **intraday and overnight**, at **21, 63 and 126** trailing observations: standardized shock, normalized historical drift, log risk ratio to the pair’s close-to-close risk, same-session leg correlation, covariance adjustment. That is 30 templates. Nine session interactions add agreement, reversal pressure, and overnight variance share at each window. Two flags identify complete-leg support and paired support. All reference moments shift by one row before rolling. The horizons are validated but the same observed context is shared across horizons; no unavailable target labels enter feature construction.

In [2]:
display(pd.DataFrame([['Intraday',15],['Overnight',15],['Cross-session interactions',9],['Structural flags',2]],columns=['Mechanism','Templates']))


## Evidence behind the hypothesis — not a promise of predictability

Blanc, Chicheportiche and Bouchaud (2013), *The fine structure of volatility feedback II*, model overnight and intraday components separately and report different volatility behavior: https://arxiv.org/abs/1309.5806. Moreira and Muir (2016/2017) motivate conditioning on risk, but their portfolio evidence is not a test of this competition: https://www.nber.org/papers/w22208.

Gorton, Hayashi and Rouwenhorst link commodity risk premiums to inventories/basis: https://www.nber.org/papers/w13249. Those mechanisms remain research leads, but no real inventory, curve or calendar feature is fabricated here from anonymous row IDs. Original competition data and target metadata define the admissible inputs for this milestone.

## Run the training-only lab

Reads only `nrows=1164` from market inputs and training labels. Screening begins at row 252 (the existing model warmup), partitions that **training** window into two halves, and reports mean daily cross-sectional candidate–target rank correlations on applicable targets. Eligibility requires nonconstant inputs and at least 60% supported coverage. Structural flags and exact duplicates are not ranked. A stable-sign, min-absolute-half-correlation ranking shortlists **at most 12** candidates. This is a shortlist for future ablations, not model promotion. Exact-duplicate screening covers these candidates plus the existing 14 normalization templates, not every historical column. No validation-outcome filter, hyperparameter search, ensemble or fitting occurs.

In [3]:
report = support.run_lab(ROOT)
print('RESULT:', report['status'])
print('Candidate templates:', report['candidate_templates'], '| New fits:', report['new_training_fits'])
print('Validation rows scored:', report['validation_rows_scored'])
display(pd.DataFrame(report['rows'])[['name','training_coverage','train_half_1_mean_ic','train_half_2_mean_ic','screen_exclusion']])
figures = support.lab_charts(report)


## 1. Supported coverage

In [4]:
figures[0][1].show(renderer='plotly_mimetype')


## 2. Training-half stability

In [5]:
figures[1][1].show(renderer='plotly_mimetype')


## 3. Shortlist for later ablations

In [6]:
figures[2][1].show(renderer='plotly_mimetype')


## 4. Candidate distributions

In [7]:
figures[3][1].show(renderer='plotly_mimetype')


## 5. Screen accounting

In [8]:
figures[4][1].show(renderer='plotly_mimetype')


## 6. Mechanism counts

In [9]:
figures[5][1].show(renderer='plotly_mimetype')


## What this does and does not establish

This produces a versioned feature tensor, names, availability metadata, source/raw hashes, training-only screening and a reproducible shortlist. **It does not establish a new validation score.** The next fitted comparison must keep the model fixed, compare additions/removals, replay saved controls, and evaluate isolated chronological periods. Do not add every shortlisted feature to a promoted model automatically. The goal is a sequence of attributable representation gains, not a target number of columns.

Raw-input learned representations, richer release-safe target/group context, point-in-time carry/inventory/fundamental data and ranking-aware loss remain separate research avenues, not assumed exhausted. Save this notebook, download its JSON/HTML, and **Stop space**. Keep feature NPZ/model files private.

In [10]:
completed = support.finish_lab(ROOT, report, figures)
print('RESULT:', completed['status'])
print('REPORT:', support.result_path(ROOT, True))
print('DASHBOARD:', completed['dashboard'])
print('Candidate checkpoint:', completed['checkpoint_path'])
print('STOP: Save notebooks and stop the SageMaker space. Do not start new fits.')
